# Notebook to plot hydrogen production and transport

Input: Solved sec-coupled PyPSA-Earth network

## Preparation

### Load packages and networks

In [ ]:
import yaml
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import pypsa
import warnings
import matplotlib.pyplot as plt
from shapely.geometry import LineString
import contextily as ctx  # Optional
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from cartopy import crs as ccrs
import cartopy.io.img_tiles as cimgt
import cartopy.feature as cfeat
import cartopy.io.shapereader as shpreader
import matplotlib.lines as mlines
import cartopy.feature as cfeature
from pathlib import Path

### Path settings

In [ ]:
# change current directory to parent folder
import os
import sys

# For Lenovo work laptop
# os.chdir('/home/marco-s/projects/alghynet')
os.chdir('/home/marco-s/projects/pypsa-earth')
# For MacBook
# os.chdir('/Users/marcoschamel/git-marco/alghynet')
sys.path.append(os.getcwd()+"/pypsa-earth/scripts")

os.path.realpath("workflow/pypsa-earth/")
# PARENT = os.path.realpath("workflow/pypsa-earth/") + "/"
PARENT = os.getcwd() + "/"

In [ ]:
PARENT

### Set scenario names and settings

In [ ]:
# Add name of config file for model run you would like to evaluate
config_name_withoutExp = "config-dz-sec-2025-forWS4-NoExp"
# config_name_withExp = "config-DZ-sec-export-h2-2050-100TWh-144H"
# config_name_withExp= 'config-DZ-sec-export-h2-2050-noSMR-0_50_100_200_300_400TWh'
# config_name_withExp= 'config-DZ-sec-2050-TestVoronoi-100clusters_200TWh'
# config_name_withExp= 'config-DZ-sec-Try_GADM_fix_200TWh'
# config_name_withExp = 'config-DZ-sec-Test_New_GADM-200TWh'
# config_name_withExp = 'config-DZ-sec-Test_Simplified_GADM'
# config_name_withExp = 'config-DZ-sec-export-h2-2050-SimplifiedNewGADM-0_50_100_200_300_400TWh'
# config_name_withExp = 'config-DZ-sec-export-h2-2050-SimplifiedNewGADM-NoH2Pipelines-0_50_100_200_300_400TWh'
# config_name_withExp = 'config-DZ-TestAugmentedLines'
# config_name_withExp = 'config-DZ-sec-export-h2-2050-AugmentedLines-newGADM-0_50_100_200_300_400_800_1000TWh'
config_name_withExp = 'config-EG-sec-2050-0_50_100_200_300_400_800_1000TWh'

config_withExp = yaml.safe_load(open(f"{PARENT}own-configs/{config_name_withExp}.yaml"))
# config_withoutExp = yaml.safe_load(open(f"{PARENT}own-configs/{config_name_withoutExp}.yaml"))

configs = {
    "withExp": config_withExp,
    #"withoutExp": config_withoutExp
}

# Choose name for current scenarioo run, will be part of output filename
# overall_name = 'noSMR-noH2Store_oldGADM'
overall_name = 'EG-v1'
# Choose wether to save figures or not
save_figs = False

### Extract settings and derive network file names

In [ ]:
settings = {}
for key, config in configs.items():
    export_values = config["export"]["h2export"]
    settings[key] = {}  # Top-level dict for each scenario

    # If export_values is a list, create a nested dict for each value
    if isinstance(export_values, list):
        for val in export_values:
            h2export_key = str(val)
            settings[key][h2export_key] = {}
            s = settings[key][h2export_key]
            s.update({
                "run_name": config["run"]["name"],
                "run_sector_name": config["run"]["sector_name"],
                "simpl": config["scenario"]["simpl"],
                "clust": config["scenario"]["clusters"],
                "ll": config["scenario"]["ll"],
                "load_scale": config["load_options"]["scale"],
                "opts": config["scenario"]["opts"],
                "sopts": config["scenario"]["sopts"],
                "planning": config["scenario"]["planning_horizons"],
                "discountrate": config["costs"]["discountrate"],
                "demand": config["scenario"]["demand"],
                "export_value": val
            })
            # String representations
            s["simpl_str"] = "_".join(map(str, s["simpl"]))
            s["clust_str"] = "_".join(map(str, s["clust"]))
            s["ll_str"] = "l" + "_".join(map(str, s["ll"]))
            s["scale_str"] = f"lc{s['load_scale']}"
            s["opts_str"] = "_".join(map(str, s["opts"]))
            s["sopts_str"] = "_".join(map(str, s["sopts"]))
            s["planning_str"] = "_".join(map(str, s["planning"]))
            s["dr_str"] = "_".join(map(str, s["discountrate"]))
            s["demand_str"] = "_".join(map(str, s["demand"]))
            s["export_str"] = f"{val}export"
            # File names and paths
            s["nc_file_name"] = (
                f"elec_s_{s['clust_str']}_ec_{s['ll_str']}_{s['opts_str']}_{s['sopts_str']}_"
                f"{s['planning_str']}_{s['dr_str']}_{s['demand_str']}_{s['export_str']}.nc"
            )
            s["nc_file_name_base"] = (
                f"elec_s_{s['clust_str']}_ec_{s['ll_str']}_{s['opts_str']}_{s['sopts_str']}_"
                f"{s['planning_str']}_{s['dr_str']}_{s['demand_str']}.nc"
            )
            s["scenario_name"] = s["run_name"]
            s["scenario_subpath"] = s["scenario_name"] + "/" if s["scenario_name"] else ""
            s["base_network_path"] = PARENT + f"results/{s['run_sector_name']}/prenetworks/{s['nc_file_name_base']}"
            s["og_network_path"] = PARENT + f"networks/{s['scenario_subpath']}elec.nc"
            s["results_path"] = PARENT + f"results/{s['run_sector_name']}/postnetworks/{s['nc_file_name']}"
            # s["results_path"] = PARENT + f"results/{s['scenario_subpath']}networks/{s['nc_file_name']}"
            s["network_path"] = PARENT + f"networks/{s['scenario_subpath']}elec.nc"
            s["regions_onshore_path"] = PARENT + f"resources/{s['scenario_subpath']}shapes/country_shapes.geojson"
            s["solar_path"] = PARENT + f"resources/{s['scenario_subpath']}renewable_profiles/profile_solar.nc"
            s["onwind_path"] = PARENT + f"resources/{s['scenario_subpath']}renewable_profiles/profile_onwind.nc"
            s["gadm_path"] = PARENT + f"resources/{s['scenario_subpath']}shapes/gadm_shapes.geojson"
    else:
        h2export_key = str(export_values)
        settings[key][h2export_key] = {}
        s = settings[key][h2export_key]
        s.update({
            "run_name": config["run"]["name"],
            "run_sector_name": config["run"]["sector_name"],
            "simpl": config["scenario"]["simpl"],
            "clust": config["scenario"]["clusters"],
            "ll": config["scenario"]["ll"],
            "load_scale": config["load_options"]["scale"],
            "opts": config["scenario"]["opts"],
            "sopts": config["scenario"]["sopts"],
            "planning": config["scenario"]["planning_horizons"],
            "discountrate": config["costs"]["discountrate"],
            "demand": config["scenario"]["demand"],
            "export_value": export_values
        })
        # String representations
        s["simpl_str"] = "_".join(map(str, s["simpl"]))
        s["clust_str"] = "_".join(map(str, s["clust"]))
        s["ll_str"] = "l" + "_".join(map(str, s["ll"]))
        s["scale_str"] = f"lc{s['load_scale']}"
        s["opts_str"] = "_".join(map(str, s["opts"]))
        s["sopts_str"] = "_".join(map(str, s["sopts"]))
        s["planning_str"] = "_".join(map(str, s["planning"]))
        s["dr_str"] = "_".join(map(str, s["discountrate"]))
        s["demand_str"] = "_".join(map(str, s["demand"]))
        s["export_str"] = f"{export_values}export"
        # File names and paths
        s["nc_file_name"] = (
            f"elec_s_{s['clust_str']}_ec_{s['ll_str']}_{s['opts_str']}_{s['sopts_str']}_"
            f"{s['planning_str']}_{s['dr_str']}_{s['demand_str']}_{s['export_str']}.nc"
        )
        s["nc_file_name_base"] = (
            f"elec_s_{s['clust_str']}_ec_{s['ll_str']}_{s['opts_str']}_{s['sopts_str']}_"
            f"{s['planning_str']}_{s['dr_str']}_{s['demand_str']}.nc"
        )
        s["scenario_name"] = s["run_name"]
        s["scenario_subpath"] = s["scenario_name"] + "/" if s["scenario_name"] else ""
        s["base_network_path"] = PARENT + f"results/{s['run_sector_name']}/prenetworks/{s['nc_file_name_base']}"
        s["og_network_path"] = PARENT + f"networks/{s['scenario_subpath']}elec.nc"
        s["results_path"] = PARENT + f"results/{s['run_sector_name']}/postnetworks/{s['nc_file_name']}"
        # s["results_path"] = PARENT + f"results/{s['scenario_subpath']}networks/{s['nc_file_name']}"
        s["network_path"] = PARENT + f"networks/{s['scenario_subpath']}elec.nc"
        s["regions_onshore_path"] = PARENT + f"resources/{s['scenario_subpath']}shapes/country_shapes.geojson"
        s["solar_path"] = PARENT + f"resources/{s['scenario_subpath']}renewable_profiles/profile_solar.nc"
        s["onwind_path"] = PARENT + f"resources/{s['scenario_subpath']}renewable_profiles/profile_onwind.nc"
        s["gadm_path"] = PARENT + f"resources/{s['scenario_subpath']}shapes/gadm_shapes.geojson"

### Import energy system networks

In [ ]:
# --- Load PyPSA networks ---

networks = {}
regions_onshore = {}
gadm_regions = {}

for key in settings:
    networks[key] = {}  # Initialize sub-dict for each scenario
    regions_onshore[key] = {}
    gadm_regions[key] = {}
    for h2export_key in settings[key]:
        s = settings[key][h2export_key]
        networks[key][h2export_key] = pypsa.Network(s["results_path"])
        regions_onshore[key][h2export_key] = gpd.read_file(s["regions_onshore_path"])
        gadm_regions[key][h2export_key] = gpd.read_file(s["gadm_path"])

## Analysis

#### Select network

In [ ]:
n = networks["withExp"]['400']

In [ ]:
regions_onshore_selec = regions_onshore["withExp"]['400']

In [ ]:
n.statistics.energy_balance(comps='Generator').divide(1e6).sort_values(ascending=False).plot(kind='bar', figsize=(10, 6), title='Energy Balance for Load Components')

### Analyse hydrogen production

#### Extract all links connected to a H2 bus

In [ ]:
h2_buses = n.buses.index[n.buses.carrier == 'H2']
links_connected_to_h2 = n.links[(n.links.bus0.isin(h2_buses)) | (n.links.bus1.isin(h2_buses))]

links_connected_to_h2[links_connected_to_h2.p_nom_opt > 10].p_nom_opt.sort_values(ascending=False).plot(kind='bar', figsize=(18, 4))

#### Extract only electrolysis

In [ ]:
# Extract electrolysis links (carrier == 'Electrolysis')
electrolysis_links = n.links[n.links.carrier == 'H2 Electrolysis']

# Get time series for these links
electrolysis_ts = n.links_t.p1[electrolysis_links.index]

# Map links to their buses
electrolysis_bus_map = electrolysis_links.bus0

# Group and sum electrolysis by bus over time
electrolysis_by_bus_ts = electrolysis_ts.T.groupby(electrolysis_bus_map).sum().T * n.snapshot_weightings.generators.mean()

# Sum over time to get total electrolysis per bus
total_electrolysis_per_bus = electrolysis_by_bus_ts.sum() / 1e6  # Convert to TWh

# Plot total electrolysis per bus
total_electrolysis_per_bus.abs().sort_values(ascending=False).plot(kind='bar', figsize=(12, 5), title='Total Electrolysis per Bus')
plt.ylabel('Hydrogen electrolysis in TWh/a')
plt.xlabel('Bus')

# Hide the top and right frame lines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add dotted horizontal lines at all y ticks
for y in plt.gca().get_yticks():
    plt.axhline(y=y, color='gray', linestyle='dotted', linewidth=0.5)
# Limit the y-axis to 350 TWh
plt.ylim(0, 300)

plt.tight_layout()
plt.show()

#### Plot hydrogen electrolysis per bus

In [ ]:
regions_onshore['withExp']['400']

In [ ]:
# Get bus coordinates for plotting
bus_coords = n.buses[["x", "y"]]

# Filter only buses present in total_electrolysis_per_bus
bus_coords_electrolysis = bus_coords.loc[total_electrolysis_per_bus.index]

# Prepare data for plotting
total_electrolysis_per_bus = total_electrolysis_per_bus.abs()  # Use absolute values for size and color
sizes = total_electrolysis_per_bus * 100  # scale for visibility

fig, ax = plt.subplots(figsize=(18, 6))

regions_onshore["withExp"]['400'].plot(ax=ax, color="lightgrey", edgecolor="black")

sc = ax.scatter(
    bus_coords_electrolysis["x"],
    bus_coords_electrolysis["y"],
    s=sizes,
    c=total_electrolysis_per_bus,
    cmap="Blues",
    alpha=0.8,
    edgecolor="k"
)

plt.colorbar(sc, ax=ax, label="Hydrogen electrolysis in TWh/a")

# Remove borders and grid
for spine in ax.spines.values():
    spine.set_visible(False)

# Remove x and y ticks and labels
ax.set_xticks([])
ax.set_yticks([]) 


plt.tight_layout()
plt.show()

### Analyse hydrogen transport

#### Extract and plot h2 pipelines

In [ ]:
n.links[n.links.carrier == "H2 pipeline repurposed"]

In [ ]:
n.buses

In [ ]:
# Filter repurposed and new hydrogen pipelines
h2_pipelines_new = n.links[n.links.carrier == "H2 pipeline"]
h2_pipelines_rep = n.links[n.links.carrier == "H2 pipeline repurposed"]

bus_coords = n.buses[["x", "y"]]

# Create LineString geometries for new pipelines
geoms_new = [
    LineString([
        bus_coords.loc[row.bus0][["x", "y"]],
        bus_coords.loc[row.bus1][["x", "y"]]
    ])
    for _, row in h2_pipelines_new.iterrows()
]

# Create LineString geometries for repurposed pipelines
geoms_rep = [
    LineString([
        bus_coords.loc[row.bus0][["x", "y"]],
        bus_coords.loc[row.bus1][["x", "y"]]
    ])
    for _, row in h2_pipelines_rep.iterrows()
]

gdf_new = gpd.GeoDataFrame(h2_pipelines_new, geometry=geoms_new, crs="EPSG:4326")
gdf_rep = gpd.GeoDataFrame(h2_pipelines_rep, geometry=geoms_rep, crs="EPSG:4326")

fig, axs = plt.subplots(1, 2, figsize=(10, 10), sharex=True, sharey=True)

regions_onshore["withExp"]['400'].plot(ax=axs[0], color="lightgrey", edgecolor="black")
gdf_rep.plot(ax=axs[0], color="orange", linewidth=2, label="Repurposed H2 Pipeline")
axs[0].set_title("Repurposed Hydrogen Pipelines")
axs[0].set_xlabel("Longitude")
axs[0].set_ylabel("Latitude")
axs[0].legend()

regions_onshore["withExp"]['400'].plot(ax=axs[1], color="lightgrey", edgecolor="black")
gdf_new.plot(ax=axs[1], color="blue", linewidth=2, label="New H2 Pipeline")
axs[1].set_title("New Hydrogen Pipelines")
axs[1].set_xlabel("Longitude")
axs[1].set_ylabel("Latitude")
axs[1].legend()

plt.tight_layout()
plt.show()

#### Plot hydrogen pipelines aaccording to their capacity

In [ ]:
buses = {}
lines = {}
capacity = {}
lines_gdf = {}
buses_gdf = {}
line_widths = {}

In [ ]:
#--- Build GeoDataFrames for repurposed H2 pipelines only ---

# Provide scenario key
key = 'withExp'
map_key = 'withExp'
h2export_key = '400'

# Initalize dictionaries
buses[map_key] = {}
lines[map_key] = {}
capacity[map_key] = {}
lines_gdf[map_key] = {}
buses_gdf[map_key] = {}
line_widths[map_key] = {}

# Filter for repurposed H2 pipelines only
h2_pipelines_rep = networks[map_key][h2export_key].links[networks[map_key][h2export_key].links.carrier == "H2 pipeline repurposed"]

# Buses
buses[map_key][h2export_key] = networks[map_key][h2export_key].buses[networks[map_key][h2export_key].buses.carrier == 'AC'].copy()
buses_gdf[map_key][h2export_key] = gpd.GeoDataFrame(
    buses[map_key][h2export_key],
    geometry=gpd.points_from_xy(buses[map_key][h2export_key].x, buses[map_key][h2export_key].y),
    crs="EPSG:4326"
)

# Pipelines
def build_pipeline_geometry(row):
    x0, y0 = networks[map_key][h2export_key].buses.loc[row.bus0, ["x", "y"]]
    x1, y1 = networks[map_key][h2export_key].buses.loc[row.bus1, ["x", "y"]]
    return LineString([(x0, y0), (x1, y1)])

pipelines_rep = h2_pipelines_rep.copy()
pipelines_rep["geometry"] = pipelines_rep.apply(build_pipeline_geometry, axis=1)
lines_gdf[map_key][h2export_key] = gpd.GeoDataFrame(pipelines_rep, geometry="geometry", crs="EPSG:4326")

# Normalize pipeline widths by capacity
if 'p_nom_opt' in lines_gdf[map_key][h2export_key].columns:
    capacity[map_key][h2export_key] = lines_gdf[map_key][h2export_key]["p_nom_opt"]
    line_widths[map_key][h2export_key] = np.interp(
        capacity[map_key][h2export_key],
        (capacity[map_key][h2export_key].min(), capacity[map_key][h2export_key].max()),
        (1.5, 9.0)
    )
else:
    print("Warning: 'p_nom_opt' not found. Using default width.")
    line_widths[map_key][h2export_key] = [1.5] * len(lines_gdf[map_key][h2export_key])

In [ ]:
# Prepare data

# Provide scenario key
key = 'withoutExp'
map_key = 'withExp'
h2export_key = '400'

# --- Plot ---
fig, ax = plt.subplots(figsize=(18, 6), subplot_kw={"projection": ccrs.PlateCarree()})

# Plot country borders
regions_onshore_selec.plot(ax=ax, color="lightgrey", edgecolor="black")

# Set extent
#ax.set_extent([-11, 15, 18, 38], crs=ccrs.PlateCarree())

# Plot lines with varying width based on capacity
for geom, width in zip(lines_gdf[map_key][h2export_key].geometry, line_widths[map_key][h2export_key]):
    ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                      linewidth=width, edgecolor="#206bc7", facecolor='none', zorder=2, linestyle='-', capstyle='round')

# Plot buses on top
ax.scatter(
    buses_gdf[map_key][h2export_key].geometry.x,
    buses_gdf[map_key][h2export_key].geometry.y,
    s=5,
    color='black',
    zorder=3,
    transform=ccrs.PlateCarree()
)

# Define representative capacities (you can tweak these)
legend_caps = [500, 1000, 5000, 10000]  # in MW

# Create matching line widths using same scaling as in plot
legend_widths = np.interp(legend_caps, (capacity[map_key][h2export_key].min(), capacity[map_key][h2export_key].max()), (1.5, 9.0))

# Create legend handles
legend_lines = [
    mlines.Line2D([], [], color='#206bc7', linewidth=lw, label=f'{cap / 1000:.1f} GW')
    for cap, lw in zip(legend_caps, legend_widths)
]

# Add legend to the plot
legend = ax.legend(handles=legend_lines, title='Pipeline capacity', loc='lower left', fontsize=13)
plt.setp(legend.get_title(), fontsize=13, fontweight='bold')

# Add legend to the plot: electtrolysis capacity


# Turn off axis, tighten layout
ax.set_axis_off()

# Add hydrogen electrolysis per bus as scatter plot
sc = ax.scatter(
    bus_coords_electrolysis["x"],
    bus_coords_electrolysis["y"],
    s=sizes,
    color="lightblue",  # fixed color for all points
    alpha=0.8,
    edgecolor="k"
)

# Add second legend showing hydrogen electrolysis capacity
# Define representative electrolysis capacities (in TWh/a)
legend_electrolysis_caps = [1, 5, 10]

# Create matching marker sizes using same scaling as in plot
legend_electrolysis_sizes = np.array(legend_electrolysis_caps) * 100  # same scaling as 'sizes'

# Create legend handles for electrolysis with spacing between entries using handler_map
legend_electrolysis = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lightblue',
               markersize=np.sqrt(s), label=f'{cap} TWh/a', alpha=0.8, markeredgecolor='k')
    for cap, s in zip(legend_electrolysis_caps, legend_electrolysis_sizes)
]

# Place the legend for electrolysis capacity
legend2 = ax.legend(handles=legend_electrolysis, title='Electrolysis capacity', loc='lower right', fontsize=13, labelspacing=1.6, borderpad=1)
plt.setp(legend2.get_title(), fontsize=13, fontweight='bold')

# Add both legends to the plot
ax.add_artist(legend)
ax.add_artist(legend2)

# Ensure the directory exists
# output_dir = f"results/graphics_general/sec/hydrogen/{h2export_key}TWh/"
# os.makedirs(output_dir, exist_ok=True)

plt.tight_layout()

# Save figure without white borders
# plt.savefig(f"{output_dir}{overall_name}_hydrogen_production-and-pipelines.pdf", bbox_inches='tight', dpi=300)

plt.show()

#### Plot utilisation of H2 pipelines

In [ ]:
h2_links = n.links.loc[n.links.carrier == 'H2 pipeline']
h2_links_rep = n.links.loc[n.links.carrier == 'H2 pipeline repurposed']

# Plot utilisation of H2 pipelines (new and repurposed)
fig, axs = plt.subplots(1, 2, figsize=(18, 6), sharex=True, sharey=True)

n.links_t.p0.loc[:, h2_links_rep.index].plot(ax=axs[0], legend=False)
axs[0].set_title("Utilisation of repurposed H2 pipelines")
axs[0].set_xlabel("Time")
axs[0].set_ylabel("Flow in MW")

n.links_t.p0.loc[:, h2_links.index].plot(ax=axs[1], legend=False)
axs[1].set_title("Utilisation of new H2 pipelines")
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Flow in MW")

plt.tight_layout()
plt.show()

Only repurposed h2 pipelines: In percentage of maximum capacity

In [ ]:
# Repurposed pipelines
utilisation_rep = n.links_t.p0[h2_links_rep.index] / n.links.loc[h2_links_rep.index, "p_nom_max"] * 100

# New pipelines
utilisation_new = n.links_t.p0[h2_links.index] / n.links.loc[h2_links.index, "p_nom_opt"] * 100

fig, ax = plt.subplots(figsize=(18, 6), sharex=True, sharey=True)

utilisation_rep.loc["2013-02"].plot(ax=ax, legend=False)
ax.set_title("Utilisation of repurposed H2 pipelines")
ax.set_xlabel("Time")
ax.set_ylabel("Utilisation (% of max capacity)")

# utilisation_new.plot(ax=axs[1], legend=False)
# axs[1].set_title("Utilisation of new H2 pipelines")
# axs[1].set_xlabel("Time")
# axs[1].set_ylabel("Utilisation (% of max capacity)")

plt.tight_layout()
plt.show()

### Analyse hydrogen demand

In [ ]:
# Extract all links where bus0 is a H2 bus and not a pipeline
h2_bus0_links = n.links[n.links.bus0.str.contains('H2') & ~n.links.carrier.str.contains('H2 pipeline')]

# Map link indices to their bus0 values
bus0_map = h2_bus0_links['bus0']

# Aggregate all H2 inflows per bus (sum over time and links)
total_h2_inflow_per_bus = n.links_t.p0[h2_bus0_links.index].T.groupby(bus0_map).sum().sum(axis=1) * n.snapshot_weightings.generators.mean() / 1e6  # Convert to TWh

# Plot hydrogen demand per bus on map
bus_coords = n.buses[["x", "y"]]
bus_coords_h2_demand = bus_coords.loc[total_h2_inflow_per_bus.index]
sizes = total_h2_inflow_per_bus * 10  # Scale sizes for better visibility

# Plot
plt.rcParams.update({'font.size': 18})

fig, ax = plt.subplots(figsize=(18, 6))

regions_onshore["withExp"]['400'].plot(ax=ax, color="lightgrey", edgecolor="black")

sc = ax.scatter(
    bus_coords_h2_demand["x"],
    bus_coords_h2_demand["y"],
    s=sizes,
    c=total_h2_inflow_per_bus,
    cmap="Reds",
    alpha=0.8,
    edgecolor="k",
    vmin=0,    # Set minimum color value
    vmax=400   # Set maximum color value
)

plt.colorbar(sc, ax=ax, label="Hydrogen demand in TWh/a")

# Remove borders and grid
for spine in ax.spines.values():
    spine.set_visible(False)

# Remove x and y ticks and labels
ax.set_xticks([])
ax.set_yticks([]) 

# Ensure the directory exists
# output_dir = f"results/graphics_general/sec/hydrogen/{h2export_key}TWh/"
# os.makedirs(output_dir, exist_ok=True)

plt.tight_layout()

# # Save figure without white borders
# plt.savefig(f"{output_dir}{overall_name}_hydrogen_demand.pdf", bbox_inches='tight', dpi=300)

plt.show()

### Analyse hydrogen export

In [ ]:
# Automatically extract the H2 export bus from the network
export_bus = n.buses.loc['H2 export bus']

# if len(export_bus) == 0:
#     # Fallback: try to find a bus with 'DZ.41_1_AC H2' if no 'export' bus found
#     export_bus = n.buses[n.buses.index == "DZ.41_1_AC H2"].index

# Extract links connected to the export bus
links_with_export_bus = n.links[n.links.bus1 == 'H2 export bus']

# Extract H2 export flow per link
h2_export_flow = n.links_t.p1[links_with_export_bus.index]

# Plot hydrogen export flow as a time series (temporal flow)
fig, ax = plt.subplots(figsize=(14, 5))
# Calculate daily sum of hydrogen export flow (in GW)
daily_h2_export_flow = h2_export_flow.sum(axis=1).resample('D').sum().multiply(n.snapshot_weightings.generators.mean()).multiply(1e-6)

# Plot daily hydrogen export flow for the selected period
daily_h2_export_flow.loc['2013'].plot(ax=ax, kind='bar', color='deepskyblue')
ax.set_title("Temporal Hydrogen Export Flow")
ax.set_ylabel("Exported Hydrogen in TWh")
ax.set_xlabel("Time")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
h2_export_flow.sum(axis=1).multiply(3).divide(1e3).plot()
plt.ylim(-150, 0)

In [ ]:
n.links[n.links.bus1 == 'H2 export bus']

## Comparative analysis

### Check scenarios

In [ ]:
for key in settings.keys():
    for h2export_key in electrolysis_links.get(key, {}).keys():
        print(f"Scenario: {key}, Export value: {h2export_key}")

### Extract electrolysis data

#### Extract electrolysis links

In [ ]:
electrolysis_links = {}

for key in settings:
    for h2export_key in settings[key]:
        # if 'electrolysis_links' not in locals():
        #     electrolysis_links = {}
        # if key not in electrolysis_links:
        #     electrolysis_links[key] = {}
        net = networks[key][h2export_key]
        mask = net.links.carrier == "H2 Electrolysis"
        electrolysis_links.setdefault(key, {})[h2export_key] = net.links.loc[mask].copy()

#### Get time series of links and map them to their buses

In [ ]:
electrolysis_results = {}

for key in electrolysis_links:
    electrolysis_results[key] = {}
    for h2export_key in electrolysis_links[key]:
        # Get electrolysis links and network
        links = electrolysis_links[key][h2export_key]
        net = networks[key][h2export_key]

        # Get time series for these links
        electrolysis_ts = net.links_t.p1[links.index]

        # Map links to their buses
        electrolysis_bus_map = links.bus0

        # Group and sum electrolysis by bus over time
        electrolysis_by_bus_ts = electrolysis_ts.T.groupby(electrolysis_bus_map).sum().T * net.snapshot_weightings.generators.mean()

        # Sum over time to get total electrolysis per bus
        total_electrolysis_per_bus = electrolysis_by_bus_ts.sum() / 1e6  # Convert to TWh

        # Save all info in nested dict
        electrolysis_results[key][h2export_key] = {
            "links": links,
            "network": net,
            "electrolysis_ts": electrolysis_ts,
            "bus_map": electrolysis_bus_map,
            "by_bus_ts": electrolysis_by_bus_ts,
            "total_per_bus": total_electrolysis_per_bus
        }

        # Plot and save
        ax = total_electrolysis_per_bus.abs().sort_values(ascending=False).plot(
            kind='bar', figsize=(12, 5), title=f'Total Electrolysis per Bus: {key}, {h2export_key}'
        )
        plt.ylabel('Hydrogen electrolysis in TWh/a')
        plt.xlabel('Bus')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        for y in ax.get_yticks():
            ax.axhline(y=y, color='gray', linestyle='dotted', linewidth=0.5)
        plt.ylim(0, 350)
        plt.tight_layout()
        # plt.savefig(f"{overall_name}_electrolysis_{key}_{h2export_key}.png", bbox_inches='tight', dpi=150)
        plt.close()


### Plot hydrogen generation for each scenario

#### Plot hydrogen electrolysis per bus

In [ ]:
# Plot hydrogen electrolysis per bus for all scenarios in one figure

num_scenarios = sum(len(electrolysis_results[key]) for key in electrolysis_results)
fig, axs = plt.subplots(1, num_scenarios, figsize=(6 * num_scenarios, 6), squeeze=False)

i = 0
for key in electrolysis_results:
    for h2export_key in electrolysis_results[key]:
        result = electrolysis_results[key][h2export_key]
        total_electrolysis_per_bus = result["total_per_bus"].abs()
        bus_coords = result["network"].buses[["x", "y"]]
        bus_coords_electrolysis = bus_coords.loc[total_electrolysis_per_bus.index]
        sizes = total_electrolysis_per_bus * 100  # scale for visibility

        ax = axs[0, i]
        # Use the correct regions_onshore for each scenario
        if key in regions_onshore and h2export_key in regions_onshore[key]:
            regions_onshore[key][h2export_key].plot(ax=ax, color="lightgrey", edgecolor="black")
        else:
            regions_onshore["withoutExp"]['0'].plot(ax=ax, color="lightgrey", edgecolor="black")

        sc = ax.scatter(
            bus_coords_electrolysis["x"],
            bus_coords_electrolysis["y"],
            s=sizes,
            #c=total_electrolysis_per_bus,
            color="lightblue",
            alpha=0.8,
            edgecolor="k"
        )
        ax.set_title(f"{key}, Export: {h2export_key}")
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_xticks([])
        ax.set_yticks([])
        i += 1

# Add a legend for electrolysis capacity
legend_electrolysis_caps = [10, 50, 100]
legend_electrolysis_sizes = np.array(legend_electrolysis_caps) * 100  # same scaling as 'sizes'
legend_electrolysis = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lightblue',
               markersize=np.sqrt(s), label=f'{cap} TWh/a', alpha=0.8, markeredgecolor='k')
    for cap, s in zip(legend_electrolysis_caps, legend_electrolysis_sizes)
]
plt.legend(handles=legend_electrolysis, title="Hydrogen production", bbox_to_anchor=(1.05, 1.15), loc='upper left', labelspacing=4.5, handletextpad=2.5, borderpad=2.5)
plt.setp(plt.gca().get_legend().get_title(), fontweight='bold')
plt.tight_layout()
plt.show()


### Analyse hydrogen transport

#### Extract and plot h2 pipelines

In [ ]:
# Plot H2 pipelines (new and repurposed) for all scenarios in settings

num_scenarios = sum(len(settings[key]) for key in settings)
fig, axs = plt.subplots(num_scenarios, 2, figsize=(12, 6 * num_scenarios), squeeze=False)

i = 0
for key in settings:
    for h2export_key in settings[key]:
        # Get network and region
        net = networks[key][h2export_key]
        region = regions_onshore[key][h2export_key]

        # Extract pipelines
        h2_pipelines_new = net.links[net.links.carrier == "H2 pipeline"]
        h2_pipelines_rep = net.links[net.links.carrier == "H2 pipeline repurposed"]
        bus_coords = net.buses[["x", "y"]]

        # Create LineString geometries for new pipelines
        geoms_new = [
            LineString([
                bus_coords.loc[row.bus0][["x", "y"]],
                bus_coords.loc[row.bus1][["x", "y"]]
            ])
            for _, row in h2_pipelines_new.iterrows()
        ]
        # Create LineString geometries for repurposed pipelines
        geoms_rep = [
            LineString([
                bus_coords.loc[row.bus0][["x", "y"]],
                bus_coords.loc[row.bus1][["x", "y"]]
            ])
            for _, row in h2_pipelines_rep.iterrows()
        ]

        gdf_new = gpd.GeoDataFrame(h2_pipelines_new, geometry=geoms_new, crs="EPSG:4326")
        gdf_rep = gpd.GeoDataFrame(h2_pipelines_rep, geometry=geoms_rep, crs="EPSG:4326")

        # Plot repurposed pipelines
        region.plot(ax=axs[i, 0], color="lightgrey", edgecolor="black")
        gdf_rep.plot(ax=axs[i, 0], color="orange", linewidth=2, label="Repurposed H2 Pipeline")
        axs[i, 0].set_title(f"{key} {h2export_key}: Repurposed Hydrogen Pipelines")
        axs[i, 0].set_xlabel("Longitude")
        axs[i, 0].set_ylabel("Latitude")
        axs[i, 0].legend()

        # Plot new pipelines
        region.plot(ax=axs[i, 1], color="lightgrey", edgecolor="black")
        gdf_new.plot(ax=axs[i, 1], color="blue", linewidth=2, label="New H2 Pipeline")
        axs[i, 1].set_title(f"{key} {h2export_key}: New Hydrogen Pipelines")
        axs[i, 1].set_xlabel("Longitude")
        axs[i, 1].set_ylabel("Latitude")
        axs[i, 1].legend()

        i += 1

plt.tight_layout()
# plt.show()

#### Plot hydrogen pipelines according to their capacity

In [ ]:
h2_pipelines_rep['withExp']['1000'].p_nom_opt.sort_values(ascending=False).plot(kind='bar', figsize=(18, 4))

##### Extract relevant data and build geometries

In [ ]:
# --- Extract and save all scenario data in nested dicts ---
scenario_keys = list(networks.keys())
h2export_keys = list(networks[scenario_keys[0]].keys())

buses = {}
lines = {}
capacity = {}
lines_gdf = {}
buses_gdf = {}
line_widths = {}
h2_pipelines_rep = {}

# Define min and max capacity for normalization
cap_min = 0  # in MW
# cap_max = 45000  # in MW
cap_max = 14000  # in MW

for scenario in scenario_keys:
    buses[scenario] = {}
    lines[scenario] = {}
    capacity[scenario] = {}
    lines_gdf[scenario] = {}
    buses_gdf[scenario] = {}
    line_widths[scenario] = {}
    h2_pipelines_rep[scenario] = {}
    for h2export_key in h2export_keys:
        # Filter for repurposed H2 pipelines only
        h2_pipelines_rep[scenario][h2export_key] = networks[scenario][h2export_key].links[
            networks[scenario][h2export_key].links.carrier == "H2 pipeline repurposed"
        ].copy()

        # Buses
        buses[scenario][h2export_key] = networks[map_key][h2export_key].buses[networks[map_key][h2export_key].buses.carrier == 'AC'].copy()
        buses_gdf[scenario][h2export_key] = gpd.GeoDataFrame(
            buses[scenario][h2export_key],
            geometry=gpd.points_from_xy(
                buses[scenario][h2export_key].x,
                buses[scenario][h2export_key].y
            ),
            crs="EPSG:4326"
        )

        # Pipelines
        def build_pipeline_geometry(row):
            x0, y0 = networks[scenario][h2export_key].buses.loc[row.bus0, ["x", "y"]]
            x1, y1 = networks[scenario][h2export_key].buses.loc[row.bus1, ["x", "y"]]
            return LineString([(x0, y0), (x1, y1)])

        pipelines_rep_temp = h2_pipelines_rep[scenario][h2export_key].copy()
        pipelines_rep = pipelines_rep_temp.copy()
        pipelines_rep["geometry"] = None  # Initialize geometry column first
        pipelines_rep["geometry"] = pipelines_rep_temp.apply(build_pipeline_geometry, axis=1)
        # Write geometry into the original dict as well
        h2_pipelines_rep[scenario][h2export_key]["geometry"] = pipelines_rep["geometry"]
        lines_gdf[scenario][h2export_key] = gpd.GeoDataFrame(pipelines_rep, geometry="geometry", crs="EPSG:4326")

        # Normalize pipeline widths by capacity
        if 'p_nom_opt' in lines_gdf[scenario][h2export_key].columns:
            capacity[scenario][h2export_key] = lines_gdf[scenario][h2export_key]["p_nom_opt"]
            line_widths[scenario][h2export_key] = np.interp(
                capacity[scenario][h2export_key],
                (cap_min, cap_max),
                (1.5, 9.0)
            )
        else:
            line_widths[scenario][h2export_key] = [1.5] * len(lines_gdf[scenario][h2export_key])

##### Plot hydrogen pipelines

In [ ]:
# --- Plot all pipelines from each scenario side by side ---
n_scenarios = len(scenario_keys) * len(h2export_keys)
fig, axes = plt.subplots(len(scenario_keys), len(h2export_keys), figsize=(8 * len(h2export_keys), 6 * len(scenario_keys)), subplot_kw={"projection": ccrs.PlateCarree()})

# Ensure axes is always 2D for consistent indexing
if len(scenario_keys) == 1 and len(h2export_keys) == 1:
    axes = np.array([[axes]])
elif len(scenario_keys) == 1:
    axes = axes[np.newaxis, :]
elif len(h2export_keys) == 1:
    axes = axes[:, np.newaxis]

for i, scenario in enumerate(scenario_keys):
    for j, h2export_key in enumerate(h2export_keys):
        ax = axes[i, j]
        ax.set_title(f"{scenario} - {h2export_key}", fontsize=16)
        # ax.add_feature(cfeature.NaturalEarthFeature('cultural', 'admin_0_countries', '50m'),
        #                facecolor='whitesmoke', edgecolor='black', zorder=0)
        # # Highlight Algeria
        # for country in reader.records():
        #     if country.attributes['SOVEREIGNT'] == 'Algeria':
        #         ax.add_geometries([country.geometry], crs=ccrs.PlateCarree(),
        #                           facecolor='lightgrey', edgecolor='black', linewidth=1, zorder=0)
        # ax.set_extent([-11, 15, 18, 38], crs=ccrs.PlateCarree())

        # Plot country borders
        regions_onshore[scenario][h2export_key].plot(ax=ax, color="lightgrey", edgecolor="black")

        # Plot pipelines for this scenario and h2export_key
        if scenario in lines_gdf and h2export_key in lines_gdf[scenario]:
            for geom, width in zip(lines_gdf[scenario][h2export_key].geometry, line_widths[scenario][h2export_key]):
                ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                                  linewidth=width, edgecolor="#206bc7", facecolor='none', zorder=2, linestyle='-', capstyle='round')

        # Plot buses
        if scenario in buses_gdf and h2export_key in buses_gdf[scenario]:
            ax.scatter(
                buses_gdf[scenario][h2export_key].geometry.x,
                buses_gdf[scenario][h2export_key].geometry.y,
                s=5,
                color='black',
                zorder=3,
                transform=ccrs.PlateCarree()
            )

        ax.set_axis_off()

# Add legend for pipeline capacity
legend_caps = [500, 1000, 5000, 10000]  # in MW
legend_widths = np.interp(legend_caps, (cap_min, cap_max), (1.5, 9.0))
legend_lines = [
    mlines.Line2D([], [], color='#206bc7', linewidth=lw, label=f'{cap / 1000:.1f} GW')
    for cap, lw in zip(legend_caps, legend_widths)
]
# Add legend below all plots, centered
fig.legend(handles=legend_lines, title='Pipeline capacity', loc='lower center', fontsize=20, ncol=len(legend_lines), bbox_to_anchor=(0.5, -0.1))
plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
plt.show()

#### Plot hydrogen pipelines and hydrogen generation

In [ ]:
# Plot pipelines and hydrogen generation per bus for each scenario side by side

fig, axes = plt.subplots(len(scenario_keys), len(settings[scenario_keys[0]]), figsize=(8 * len(settings[scenario_keys[0]]), 6 * len(scenario_keys)), subplot_kw={"projection": ccrs.PlateCarree()})

# Ensure axes is always 2D for consistent indexing
if len(scenario_keys) == 1 and len(settings[scenario_keys[0]]) == 1:
    axes = np.array([[axes]])
elif len(scenario_keys) == 1:
    axes = axes[np.newaxis, :]
elif len(settings[scenario_keys[0]]) == 1:
    axes = axes[:, np.newaxis]

for i, scenario in enumerate(scenario_keys):
    for j, h2export_key in enumerate(settings[scenario].keys()):
        ax = axes[i, j]
        ax.set_title(f"Hydrogen export: {h2export_key} TWh/a", fontsize=20)
        # Country background
        ax.add_feature(cfeature.NaturalEarthFeature('cultural', 'admin_0_countries', '50m'),
                       facecolor='whitesmoke', edgecolor='black', zorder=0)
        # Highlight Algeria
        for country in reader.records():
            if country.attributes['SOVEREIGNT'] == 'Algeria':
                ax.add_geometries([country.geometry], crs=ccrs.PlateCarree(),
                                  facecolor='lightgrey', edgecolor='black', linewidth=1, zorder=0)
        ax.set_extent([-11, 15, 18, 38], crs=ccrs.PlateCarree())

        # Plot pipelines for this scenario and h2export_key
        if scenario in lines_gdf and h2export_key in lines_gdf[scenario]:
            for geom, width in zip(lines_gdf[scenario][h2export_key].geometry, line_widths[scenario][h2export_key]):
                ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                                  linewidth=width, edgecolor="#206bc7", facecolor='none', zorder=2, linestyle='-', capstyle='round')

        # Plot buses
        if scenario in buses_gdf and h2export_key in buses_gdf[scenario]:
            ax.scatter(
                buses_gdf[scenario][h2export_key].geometry.x,
                buses_gdf[scenario][h2export_key].geometry.y,
                s=5,
                color='black',
                zorder=3,
                transform=ccrs.PlateCarree()
            )

        # Plot hydrogen generation per bus
        result = electrolysis_results[scenario][h2export_key]
        total_electrolysis_per_bus = result["total_per_bus"].abs()
        bus_coords = result["network"].buses[["x", "y"]]
        bus_coords_electrolysis = bus_coords.loc[total_electrolysis_per_bus.index]
        sizes = total_electrolysis_per_bus * 100  # scale for visibility

        ax.scatter(
            bus_coords_electrolysis["x"],
            bus_coords_electrolysis["y"],
            s=sizes,
            color="lightblue",
            alpha=0.8,
            edgecolor="k",
            zorder=4,
            transform=ccrs.PlateCarree()
        )

        ax.set_axis_off()

# -----------------------------
# Legend 1: Pipeline capacity
legend_caps = [1000, 10000, 20000, 40000]  # in MW
legend_widths = np.interp(legend_caps, (cap_min, cap_max), (1.5, 9.0))
legend_lines = [
    mlines.Line2D([], [], color='#206bc7', linewidth=lw, label=f'{cap / 1000:.1f} GW')
    for cap, lw in zip(legend_caps, legend_widths)
]

legend1 = fig.legend(
    handles=legend_lines,
    title="Pipeline capacity",
    loc="upper center",           # anchor refers to top-center
    bbox_to_anchor=(0.3, 0),   # (x, y) in figure coords
    fontsize=20,
    ncol=len(legend_lines),
    borderpad=2.4,
    labelspacing=4.0,
    title_fontproperties={"weight": "bold"}
)

# -----------------------------
# Legend 2: Hydrogen production
legend_electrolysis_caps = [10, 50, 100]
legend_electrolysis_sizes = np.array(legend_electrolysis_caps) * 100
legend_electrolysis = [
    plt.Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor="lightblue",
        markersize=np.sqrt(s),
        label=label,
        alpha=0.8,
        markeredgecolor="k",
    )
    for s, label in zip(legend_electrolysis_sizes, [
        "10 TWh/a   ",
        "    50 TWh/a      ",
        "        100 TWh/a   "
    ])
]

legend2 = fig.legend(
    handles=legend_electrolysis,
    title="Hydrogen production",
    loc="upper center",
    bbox_to_anchor=(0.7, 0),
    fontsize=20,
    ncol=len(legend_electrolysis),
    borderpad=2.4,
    labelspacing=4.0,
    handletextpad=0.9,
    title_fontproperties={"weight": "bold"}
)

plt.tight_layout()

# Ensure the directory exists
output_dir = f"results/graphics_general/sec-NoH2Store/hydrogen/"
os.makedirs(output_dir, exist_ok=True)

# Save figure without white borders
if save_figs:
    # Build filename with all scenario keys and subkeys
    scenario_str = "_".join(scenario_keys)
    subkeys_str = "_".join([str(subkey) for subkey in settings[scenario_keys[0]].keys()])
    filename = f"{output_dir}{overall_name}_hydrogen_production-and-pipelines_{scenario_str}_{subkeys_str}.pdf"
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    filename_png = filename.replace('.pdf', '.png')
    plt.savefig(filename_png, bbox_inches='tight', dpi=300)

plt.show()


### Analyse and plot powerlines

#### Extract power lines and build geometries

In [ ]:
# --- Extract and save all scenario data in nested dicts ---
scenario_keys = list(networks.keys())
h2export_keys = list(networks[scenario_keys[0]].keys())

elec_buses = {}
elec_lines = {}
elec_capacity = {}
elec_lines_gdf = {}
elec_buses_gdf = {}
elec_line_widths = {}
powerlines = {}

# Define min and max capacity for normalization
cap_min = 0  # in MW
cap_max = 60000  # in MW

for scenario in scenario_keys:
    elec_buses[scenario] = {}
    elec_lines[scenario] = {}
    elec_capacity[scenario] = {}
    elec_lines_gdf[scenario] = {}
    elec_buses_gdf[scenario] = {}
    elec_line_widths[scenario] = {}
    powerlines[scenario] = {}
    for h2export_key in h2export_keys:
        # Filter for repurposed H2 pipelines only
        powerlines[scenario][h2export_key] = networks[scenario][h2export_key].lines.copy()

        # Buses
        elec_buses[scenario][h2export_key] = networks[scenario][h2export_key].buses.copy()
        elec_buses_gdf[scenario][h2export_key] = gpd.GeoDataFrame(
            elec_buses[scenario][h2export_key],
            geometry=gpd.points_from_xy(
                elec_buses[scenario][h2export_key].x,
                elec_buses[scenario][h2export_key].y
            ),
            crs="EPSG:4326"
        )

        # Pipelines
        def build_powerline_geometry(row):
            x0, y0 = networks[scenario][h2export_key].buses.loc[row.bus0, ["x", "y"]]
            x1, y1 = networks[scenario][h2export_key].buses.loc[row.bus1, ["x", "y"]]
            return LineString([(x0, y0), (x1, y1)])

        powerlines_temp = powerlines[scenario][h2export_key].copy()
        # powerlines_rep = powerlines_temp.copy()
        powerlines_temp["geometry"] = None  # Initialize geometry column first
        powerlines_temp["geometry"] = powerlines_temp.apply(build_powerline_geometry, axis=1)
        # Write geometry into the original dict as well
        powerlines[scenario][h2export_key]["geometry"] = powerlines_temp["geometry"]
        elec_lines_gdf[scenario][h2export_key] = gpd.GeoDataFrame(powerlines_temp, geometry="geometry", crs="EPSG:4326")

        # Normalize pipeline widths by capacity
        if 's_nom_opt' in elec_lines_gdf[scenario][h2export_key].columns:
            elec_capacity[scenario][h2export_key] = elec_lines_gdf[scenario][h2export_key]["s_nom_opt"]
            elec_line_widths[scenario][h2export_key] = np.interp(
                elec_capacity[scenario][h2export_key],
                (cap_min, cap_max),
                (1.5, 9.0)
            )
        else:
            elec_line_widths[scenario][h2export_key] = [1.5] * len(elec_lines_gdf[scenario][h2export_key])

In [ ]:
elec_capacity['withExp']['400'].sort_values(ascending=False).plot(kind='bar', figsize=(12, 5), title='Electricity line capacities in MW: withExp')

#### Plot powerlines

In [ ]:
# Plot powerlines for each scenario side by side

fig, axes = plt.subplots(len(scenario_keys), len(settings[scenario_keys[0]]), figsize=(8 * len(settings[scenario_keys[0]]), 6 * len(scenario_keys)), subplot_kw={"projection": ccrs.PlateCarree()})

# Ensure axes is always 2D for consistent indexing
if len(scenario_keys) == 1 and len(settings[scenario_keys[0]]) == 1:
    axes = np.array([[axes]])
elif len(scenario_keys) == 1:
    axes = axes[np.newaxis, :]
elif len(settings[scenario_keys[0]]) == 1:
    axes = axes[:, np.newaxis]

for i, scenario in enumerate(scenario_keys):
    for j, h2export_key in enumerate(settings[scenario].keys()):
        ax = axes[i, j]
        ax.set_title(f"Hydrogen export: {h2export_key} TWh/a", fontsize=20)
        # Country background
        ax.add_feature(cfeature.NaturalEarthFeature('cultural', 'admin_0_countries', '50m'),
                       facecolor='whitesmoke', edgecolor='black', zorder=0)
        
        # Highlight Algeria separately
        shp_filename = shpreader.natural_earth(resolution='50m', category='cultural', name='admin_0_countries')
        reader = shpreader.Reader(shp_filename)
        countries = reader.records()

        # Highlight Algeria
        for country in reader.records():
            if country.attributes['SOVEREIGNT'] == 'Algeria':
                ax.add_geometries([country.geometry], crs=ccrs.PlateCarree(),
                                  facecolor='lightgrey', edgecolor='black', linewidth=1, zorder=0)
        ax.set_extent([-11, 15, 18, 38], crs=ccrs.PlateCarree())

        # Plot pipelines for this scenario and h2export_key
        if scenario in elec_lines_gdf and h2export_key in elec_lines_gdf[scenario]:
            for geom, width in zip(elec_lines_gdf[scenario][h2export_key].geometry, elec_line_widths[scenario][h2export_key]):
                ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                                  linewidth=width, edgecolor="#206bc7", facecolor='none', zorder=2, linestyle='-', capstyle='round')

        # Plot buses
        if scenario in elec_buses_gdf and h2export_key in elec_buses_gdf[scenario]:
            ax.scatter(
                elec_buses_gdf[scenario][h2export_key].geometry.x,
                elec_buses_gdf[scenario][h2export_key].geometry.y,
                s=5,
                color='black',
                zorder=3,
                transform=ccrs.PlateCarree()
            )

        # Plot hydrogen generation per bus
        result = electrolysis_results[scenario][h2export_key]
        total_electrolysis_per_bus = result["total_per_bus"].abs()
        bus_coords = result["network"].buses[["x", "y"]]
        bus_coords_electrolysis = bus_coords.loc[total_electrolysis_per_bus.index]
        sizes = total_electrolysis_per_bus * 100  # scale for visibility

        ax.scatter(
            bus_coords_electrolysis["x"],
            bus_coords_electrolysis["y"],
            s=sizes,
            color="lightblue",
            alpha=0.8,
            edgecolor="k",
            zorder=4,
            transform=ccrs.PlateCarree()
        )

        ax.set_axis_off()

# -----------------------------
# Legend 1: Pipeline capacity
legend_caps = [1000, 10000, 30000, 60000]  # in MW
legend_widths = np.interp(legend_caps, (cap_min, cap_max), (1.5, 9.0))
legend_lines = [
    mlines.Line2D([], [], color='#206bc7', linewidth=lw, label=f'{cap / 1000:.1f} GW')
    for cap, lw in zip(legend_caps, legend_widths)
]

legend1 = fig.legend(
    handles=legend_lines,
    title="Pipeline capacity",
    loc="upper center",           # anchor refers to top-center
    bbox_to_anchor=(0.3, 0),   # (x, y) in figure coords
    fontsize=20,
    ncol=len(legend_lines),
    borderpad=2.4,
    labelspacing=4.0,
    title_fontproperties={"weight": "bold"}
)

# -----------------------------
# Legend 2: Hydrogen production
legend_electrolysis_caps = [10, 50, 100]
legend_electrolysis_sizes = np.array(legend_electrolysis_caps) * 100
legend_electrolysis = [
    plt.Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor="lightblue",
        markersize=np.sqrt(s),
        label=label,
        alpha=0.8,
        markeredgecolor="k",
    )
    for s, label in zip(legend_electrolysis_sizes, [
        "10 TWh/a   ",
        "    50 TWh/a      ",
        "        100 TWh/a   "
    ])
]

legend2 = fig.legend(
    handles=legend_electrolysis,
    title="Hydrogen production",
    loc="upper center",
    bbox_to_anchor=(0.7, 0),
    fontsize=20,
    ncol=len(legend_electrolysis),
    borderpad=2.4,
    labelspacing=4.0,
    handletextpad=0.9,
    title_fontproperties={"weight": "bold"}
)

plt.tight_layout()

# Ensure the directory exists
output_dir = f"results/graphics_general/sec-NoH2Store/hydrogen+electricity/"
os.makedirs(output_dir, exist_ok=True)

# Save figure without white borders
if save_figs:
    # Build filename with all scenario keys and subkeys
    scenario_str = "_".join(scenario_keys)
    subkeys_str = "_".join([str(subkey) for subkey in settings[scenario_keys[0]].keys()])
    filename = f"{output_dir}{overall_name}_powerlines_{scenario_str}_{subkeys_str}.pdf"
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    filename_png = filename.replace('.pdf', '.png')
    plt.savefig(filename_png, bbox_inches='tight', dpi=300)

plt.show()


In [ ]:
elec_capacity['withExp']['400'].sort_values(ascending=False).head(10)

In [ ]:
powerlines['withExp']['400'].s_nom_opt

In [ ]:
n.buses[n.buses.carrier == 'AC']

In [ ]:
pipelines_rep.p_nom_opt.sort_values(ascending=False).head(10) / 1e3  # in GW

### Analyse hydrogen demand

In [ ]:
# Extract and plot spatially resolved hydrogen demand for all scenarios

demand_results = {}

num_scenarios = sum(len(settings[key]) for key in settings)
fig, axs = plt.subplots(1, num_scenarios, figsize=(6 * num_scenarios, 6), squeeze=False)

i = 0
for key in settings:
    for h2export_key in settings[key]:
        net = networks[key][h2export_key]
        region = regions_onshore[key][h2export_key]

        # Extract all links where bus0 is a H2 bus and not a pipeline
        h2_bus0_links = net.links[net.links.bus0.str.contains('H2') & ~net.links.carrier.str.contains('H2 pipeline')]
        bus0_map = h2_bus0_links['bus0']

        # Aggregate all H2 inflows per bus (sum over time and links)
        total_h2_inflow_per_bus = net.links_t.p0[h2_bus0_links.index].T.groupby(bus0_map).sum().sum(axis=1) * net.snapshot_weightings.generators.mean() / 1e6  # TWh

        # Save results
        demand_results.setdefault(key, {})[h2export_key] = total_h2_inflow_per_bus

        # Prepare for plotting
        bus_coords = net.buses[["x", "y"]]
        bus_coords_h2_demand = bus_coords.loc[total_h2_inflow_per_bus.index]
        sizes = total_h2_inflow_per_bus * 10

        ax = axs[0, i]
        region.plot(ax=ax, color="lightgrey", edgecolor="black")
        sc = ax.scatter(
            bus_coords_h2_demand["x"],
            bus_coords_h2_demand["y"],
            s=sizes,
            #c=total_h2_inflow_per_bus,
            color="#f96d6d",  # light red hex code
            alpha=0.8,
            edgecolor="k",
            vmin=0,
            vmax=400
        )
        ax.set_title(f"Hydrogen export: {h2export_key} TWh/a", fontsize=20)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_xticks([])
        ax.set_yticks([])
        i += 1

# plt.colorbar(sc, ax=axs[0, -1], label="Hydrogen demand in TWh/a")
# plt.tight_layout()
# plt.show()

# Add a legend for hydrogen demand
legend_demand_caps = [50, 100, 200]
legend_demand_sizes = np.array(legend_demand_caps) * 10  # same scaling as 'sizes'
legend_demand = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff9999',
               markersize=np.sqrt(s), label=f'{cap} TWh/a', alpha=0.8, markeredgecolor='k')
    for cap, s in zip(legend_demand_caps, legend_demand_sizes)
]
legend = fig.legend(
    handles=legend_demand,
    title="Hydrogen demand",
    loc="lower center",
    bbox_to_anchor=(0.5, -0.32),
    labelspacing=2,
    handletextpad=1.5,
    borderpad=1.6,
    ncol=len(legend_demand),
    fontsize=20
)
plt.setp(legend.get_title(), fontweight='bold')
plt.tight_layout()

# Ensure the directory exists
output_dir = f"results/graphics_general/sec-NoH2Store/hydrogen/"
os.makedirs(output_dir, exist_ok=True)

# Save figure without white borders
if save_figs:
    # Build filename with all scenario keys and subkeys
    scenario_str = "_".join(scenario_keys)
    subkeys_str = "_".join([str(subkey) for subkey in settings[scenario_keys[0]].keys()])
    filename = f"{output_dir}{overall_name}_hydrogen_demand_{scenario_str}_{subkeys_str}.pdf"
    plt.savefig(filename, bbox_inches='tight', dpi=300)

plt.show()

# Optionally save results to file
# import pickle
# with open("demand_results.pkl", "wb") as f:
#     pickle.dump(demand_results, f)

## ADD-IN: Further investigations

### Check all sources of H2 (incl. SMR)

In [ ]:
n = networks['withExp']['1000']
all_h2 = n.links[
    n.links.bus1.str.contains("H2")
    & ~n.links.carrier.str.contains("H2 pipeline")
    & ~n.links.index.str.contains("H2 export")
]
all_h2_prod = n.links_t.p1[all_h2.index]
all_h2_prod_sum = all_h2_prod.sum(axis=0).multiply(n.snapshot_weightings.generators.mean()).abs()
all_h2_prod_sum.loc[all_h2_prod_sum > 1e5].abs().sort_values(ascending=False).divide(1e6).plot(figsize=(12, 5), kind='bar')
plt.ylabel("Hydrogen production in TWh/a")
plt.title("Total Hydrogen production (all links)")

### Check emissions

#### Analyse CO2 to store

In [ ]:
# Extract all co2 links to co2 store
co2_links = n.links[
    (
        n.links.bus2.str.contains('co2 stored') |
        n.links.bus3.str.contains('co2 stored') |
        n.links.bus4.str.contains('co2 stored')
    ) & (n.links.carrier != "co2 pipeline")
]
# For each link, find which bus (bus1, bus2, bus3) contains 'co2 stored'
emissions = []
link_names = []

for idx, row in co2_links.iterrows():
    # Check which bus contains 'co2 stored'
    bus_col = None
    p_col = None
    if 'co2 stored' in str(row.get('bus1', '')):
        bus_col = 'bus1'
        p_col = 'p1'
    elif 'co2 stored' in str(row.get('bus2', '')):
        bus_col = 'bus2'
        p_col = 'p2'
    elif 'co2 stored' in str(row.get('bus3', '')):
        bus_col = 'bus3'
        p_col = 'p3'
    elif 'co2 stored' in str(row.get('bus4', '')):
        bus_col = 'bus4'
        p_col = 'p4'
    else:
        continue  # skip if not found

    # Sum up the yearly emissions for this link
    if p_col in n.links_t:
        emission = n.links_t[p_col][idx].sum() * n.snapshot_weightings.generators.mean() / 1e6  # Mt/a
        emissions.append(emission)
        link_names.append(idx)

# Create a Series for plotting
emissions_series = pd.Series(emissions, index=link_names)
emissions_series = emissions_series.sort_values(ascending=False)

# Calculate total emissions to storage
total_emissions = emissions_series.sum()
print(f"Total CO2 emissions to storage: {abs(total_emissions):.2f} Mt/a")

# Plot
emissions_series[emissions_series < -0.1].abs().plot(kind='bar', figsize=(12, 5))
plt.ylabel('CO2 emissions to CO2 storage in Mt/a')
plt.xlabel('Link')
plt.tight_layout()
plt.show()

#### Analyse cO2 to atmosphere

In [ ]:
# Extract all co2 links
co2_links = n.links[
    (
        n.links.bus2.str.contains('co2 atmosphere') |
        n.links.bus3.str.contains('co2 atmosphere') |
        n.links.bus4.str.contains('co2 atmosphere')
    ) & (n.links.carrier != "co2 pipeline")
]
# For each link, find which bus (bus1, bus2, bus3) contains 'co2 atmosphere'
emissions = []
link_names = []

for idx, row in co2_links.iterrows():
    # Check which bus contains 'co2 atmosphere'
    bus_col = None
    p_col = None
    if 'co2 atmosphere' in str(row.get('bus1', '')):
        bus_col = 'bus1'
        p_col = 'p1'
    elif 'co2 atmosphere' in str(row.get('bus2', '')):
        bus_col = 'bus2'
        p_col = 'p2'
    elif 'co2 atmosphere' in str(row.get('bus3', '')):
        bus_col = 'bus3'
        p_col = 'p3'
    elif 'co2 atmosphere' in str(row.get('bus4', '')):
        bus_col = 'bus4'
        p_col = 'p4'
    else:
        continue  # skip if not found

    # Sum up the yearly emissions for this link
    if p_col in n.links_t:
        emission = n.links_t[p_col][idx].sum() * n.snapshot_weightings.generators.mean() / 1e6  # Mt/a
        emissions.append(emission)
        link_names.append(idx)

# Create a Series for plotting
emissions_series = pd.Series(emissions, index=link_names)
emissions_series = emissions_series.sort_values(ascending=False)

# Plot
emissions_series[emissions_series < -0.01].plot(kind='bar', figsize=(12, 5))
plt.ylabel('CO2 emissions to atmosphere in Mt/a')
plt.xlabel('Link')
plt.tight_layout()
plt.show()

### Check solar generation data

#### Calculate maximum vs. optimised generation

In [ ]:
# Extract solar generators
solar_gens = n.generators[n.generators.carrier == "solar"]

# Extract actual generation time series for these generators
solar_gens_actual = n.generators_t.p[solar_gens.index]

# Extract maximum capacity (p_nom_max) for these generators
solar_gens_max = n.generators.loc[solar_gens.index, "p_nom_max"]

# Extract optimized capacity (p_nom_opt) for these generators
solar_gens_opt = n.generators.loc[solar_gens.index, "p_nom_opt"]

# Extract maximum potential (p_max_pu) for these generators
solar_gens_max_pu = n.generators_t.p_max_pu[solar_gens.index]

# Calculate possible generation time series
# solar_gens_max: (47,)
# solar_gens_max_pu: (2920, 47)
# We want (2920, 47): time x generators
# Calculate theoretical maximum generation time series (using p_nom_max)
theoretical_max_gen = solar_gens_max_pu.values * solar_gens_max.values[np.newaxis, :]

# Calculate actual maximum generation time series (using p_nom_opt)
actual_max_gen = solar_gens_max_pu.values * solar_gens_opt.values[np.newaxis, :]

# Convert to DataFrames
theoretical_max_gen_df = pd.DataFrame(theoretical_max_gen, index=solar_gens_max_pu.index, columns=solar_gens.index)
actual_max_gen_df = pd.DataFrame(actual_max_gen, index=solar_gens_max_pu.index, columns=solar_gens.index)

# Calculate annual generation per generator (TWh/a)
theoretical_max_per_gen = theoretical_max_gen_df.multiply(n.snapshot_weightings.generators.mean()).sum() / 1e6
actual_max_per_gen = actual_max_gen_df.multiply(n.snapshot_weightings.generators.mean()).sum() / 1e6

# Combine into one DataFrame for plotting
plot_df = pd.DataFrame({
    "Theoretical max": theoretical_max_per_gen,
    "Actual max": actual_max_per_gen
})

# Plot as grouped bar chart
plot_df.sort_values("Theoretical max", ascending=False).plot(kind="bar", figsize=(14, 6))
plt.ylabel("Solar generation in TWh/a")
plt.gca().yaxis.set_major_formatter('{:,.0f}'.format)
plt.title("Theoretical vs Actual Maximum Solar Generation per Generator")
plt.tight_layout()
plt.show()

#### Plot maximum PV capacity per bus

In [ ]:
solar_gens.index

In [ ]:
# Create geodataframe with p_nom_max, bus names, and coodinates
# Ensure the buses are aligned and unique
bus_info = n.buses.loc[solar_gens.bus].reset_index(drop=True)
solar_gens_geo = gpd.GeoDataFrame({
    'generator': solar_gens.index,
    "bus": solar_gens.bus.values,
    "p_nom_max": solar_gens.p_nom_max.values,
    "p_nom_opt": solar_gens.p_nom_opt.values,
    "geometry": gpd.points_from_xy(
        bus_info.x.values,
        bus_info.y.values
    ),
    "country": bus_info.country.values
}, crs="EPSG:4326")

# Plot solar capacity p_nom_max on map
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"projection": ccrs.PlateCarree()})
# ax.set_title("Solar Generator Capacity (p_nom_max)", fontsize=20)
ax.add_feature(cfeature.NaturalEarthFeature('cultural', 'admin_0_countries', '50m'),
               facecolor='whitesmoke', edgecolor='black', zorder=0)
# Highlight Algeria
for country in reader.records():
    if country.attributes['SOVEREIGNT'] == 'Algeria':
        ax.add_geometries([country.geometry], crs=ccrs.PlateCarree(),
                          facecolor='lightgrey', edgecolor='black', linewidth=1, zorder=0)
ax.set_extent([-11, 15, 18, 38], crs=ccrs.PlateCarree())
# Aggregate p_nom_max per bus region
region_p_nom_max_GW = solar_gens_geo.groupby("bus")["p_nom_max"].sum().divide(1e3)  # in GW

# Merge with gadm_shapes using GADM ID (partial match)
# Assume gadm_shapes has a column 'GADM_ID' and regions_p_nom_max index contains GADM IDs (possibly with extra info)

def match_gadm_id(row, region_index):
    # Find first region index that contains the GADM ID as substring
    matches = [idx for idx in region_index if row['GADM_ID'] in idx]
    if matches:
        return region_p_nom_max_GW[matches[0]]
    else:
        return 0

gadm_regions_selected = gadm_regions['withExp']['400']
gadm_regions_selected["p_nom_max"] = gadm_regions_selected.apply(lambda row: match_gadm_id(row, region_p_nom_max_GW.index), axis=1)
bus_regions = gadm_regions_selected.copy()

# Plot bus regions colored by p_nom_max
mappable = bus_regions.plot(
    ax=ax,
    column="p_nom_max",
    cmap="YlOrRd",
    legend=False,
    edgecolor="black",
    linewidth=0.5,
    zorder=2
)

# Add colorbar (fix: use the first PolyCollection from mappable.collections)
sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(vmin=bus_regions["p_nom_max"].min(), vmax=bus_regions["p_nom_max"].max()))
sm._A = []  # Dummy array for ScalarMappable
cbar = fig.colorbar(sm, ax=ax, orientation='vertical', fraction=0.036, pad=0.04)
cbar.set_label('Total solar capacity in GW', fontsize=18)
cbar.ax.yaxis.set_major_formatter('{:,.0f}'.format)

plt.tight_layout()
plt.show()

#### Plot PV solar generation per bus

In [ ]:
# Plot solar capacity p_nom_opt on map
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"projection": ccrs.PlateCarree()})
# ax.set_title("Solar Generator Capacity (p_nom_opt)", fontsize=20)
# ax.add_feature(cfeature.NaturalEarthFeature('cultural', 'admin_0_countries', '50m'),
#                facecolor='whitesmoke', edgecolor='black', zorder=0)
# # Highlight Algeria
# for country in reader.records():
#     if country.attributes['SOVEREIGNT'] == 'Algeria':
#         ax.add_geometries([country.geometry], crs=ccrs.PlateCarree(),
#                           facecolor='lightgrey', edgecolor='black', linewidth=1, zorder=0)
# ax.set_extent([-11, 15, 18, 38], crs=ccrs.PlateCarree())
# Aggregate p_nom_opt per bus region
region_p_nom_opt_GW = solar_gens_geo.groupby("bus")["p_nom_opt"].sum().divide(1e3)  # in GW

# Merge with gadm_shapes using GADM ID (partial match)
# Assume gadm_shapes has a column 'GADM_ID' and regions_p_nom_max index contains GADM IDs (possibly with extra info)

def match_gadm_id(row, region_index):
    # Find first region index that contains the GADM ID as substring
    matches = [idx for idx in region_index if row['GADM_ID'] in idx]
    if matches:
        return region_p_nom_opt_GW[matches[0]]
    else:
        return 0

gadm_regions_selected = gadm_regions['withExp']['400']
gadm_regions_selected["p_nom_opt"] = gadm_regions_selected.apply(lambda row: match_gadm_id(row, region_p_nom_opt_GW.index), axis=1)
bus_regions = gadm_regions_selected.copy()

# Plot bus regions colored by p_nom_opt
mappable = bus_regions.plot(
    ax=ax,
    column="p_nom_opt",
    cmap="YlOrRd",
    legend=False,
    edgecolor="black",
    linewidth=0.5,
    zorder=2
)

# Add colorbar (fix: use the first PolyCollection from mappable.collections)
sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(vmin=bus_regions["p_nom_opt"].min(), vmax=bus_regions["p_nom_opt"].max()))
sm._A = []  # Dummy array for ScalarMappable
cbar = fig.colorbar(sm, ax=ax, orientation='vertical', fraction=0.036, pad=0.04)
cbar.set_label('Installed solar capacity in GW', fontsize=18)
cbar.ax.yaxis.set_major_formatter('{:,.0f}'.format)

plt.tight_layout()
plt.show()

#### Calculate actual full load hours of each generator

In [ ]:
# Calculate full load hours for solar generators
solar_gens_flh = solar_gens_actual.multiply(n.snapshot_weightings.generators.mean()).sum() / solar_gens_opt

fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot full load hours
solar_gens_flh.sort_values(ascending=False).plot(kind="bar", figsize=(14, 6), color="lightblue", ax=ax1)
ax1.set_ylabel("Full Load Hours (FLH)")
# ylabel to include thousand separator
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.ylim(0, 1800)
# plt.title("Full Load Hours per Solar Generator") 

# Include actual generation in second y-axis
ax2 = plt.gca().twinx()
solar_gens_actual.sum().multiply(n.snapshot_weightings.generators.mean()).divide(1e6).sort_values(ascending=False).plot(kind="line", color="purple", marker="o", ax=ax2)
ax2.set_ylabel("Actual Generation in TWh/a", color="black")
ax2.tick_params(axis='y', labelcolor="black")
# Remove top border for both axes
ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)
# Rename x-axis ticks to include only bus names
# Get current xticklabels and rename them by removing "_AC solar"
xticklabels = [label.get_text().replace("_AC solar", "") for label in ax1.get_xticklabels()]
ax1.set_xticklabels(xticklabels, ha='right')

fig.tight_layout()
plt.show()


#### Plot full load hours per bus

In [ ]:
# Add FLH to solar_gens_geo by matching index of solar_gens_flh with generator in solar_gens_geo
# Ensure fallback if FLH exists already
if "FLH" in solar_gens_geo.columns:
    solar_gens_geo = solar_gens_geo.drop(columns=["FLH"])
solar_gens_geo = solar_gens_geo.set_index("generator").join(solar_gens_flh.rename("FLH")).reset_index()
#solar_gens_geo = solar_gens_geo.set_index("bus").join(solar_gens_flh.rename("FLH")).reset_index()
solar_gens_geo.head()

In [ ]:
# Plot solar capacity p_nom_opt on map
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"projection": ccrs.PlateCarree()})
# ax.set_title("Solar Generator Capacity (p_nom_opt)", fontsize=20)
# ax.add_feature(cfeature.NaturalEarthFeature('cultural', 'admin_0_countries', '50m'),
#                facecolor='whitesmoke', edgecolor='black', zorder=0)
# # Highlight Algeria
# for country in reader.records():
#     if country.attributes['SOVEREIGNT'] == 'Algeria':
#         ax.add_geometries([country.geometry], crs=ccrs.PlateCarree(),
#                           facecolor='lightgrey', edgecolor='black', linewidth=1, zorder=0)
# ax.set_extent([-11, 15, 18, 38], crs=ccrs.PlateCarree())
# Aggregate p_nom_opt per bus region
region_FLH = solar_gens_geo.groupby("bus")["FLH"].sum()  # in hours

# Merge with gadm_shapes using GADM ID (partial match)
# Assume gadm_shapes has a column 'GADM_ID' and regions_p_nom_max index contains GADM IDs (possibly with extra info)

def match_gadm_id(row, region_index):
    # Find first region index that contains the GADM ID as substring
    matches = [idx for idx in region_index if row['GADM_ID'] in idx]
    if matches:
        return region_FLH[matches[0]]
    else:
        return 0

gadm_regions_selected = gadm_regions['withExp']['400']
gadm_regions_selected["FLH"] = gadm_regions_selected.apply(lambda row: match_gadm_id(row, region_FLH.index), axis=1)
bus_regions = gadm_regions_selected.copy()

# Plot bus regions colored by FLH
mappable = bus_regions.plot(
    ax=ax,
    column="FLH",
    cmap="YlOrRd",
    legend=False,
    edgecolor="black",
    linewidth=0.5,
    zorder=2
)

# Add colorbar (fix: use the first PolyCollection from mappable.collections)
# sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(vmin=bus_regions["FLH"].min(), vmax=bus_regions["FLH"].max()))
sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(vmin=400, vmax=bus_regions["FLH"].max()))
sm._A = []  # Dummy array for ScalarMappable
cbar = fig.colorbar(sm, ax=ax, orientation='vertical', fraction=0.036, pad=0.04)
cbar.set_label('Full load hours (FLH)', fontsize=18)
cbar.ax.yaxis.set_major_formatter('{:,.0f}'.format)

plt.tight_layout()

# Ensure the directory exists
output_dir = f"results/graphics_general/sec-NoH2Store/electricity/"
os.makedirs(output_dir, exist_ok=True)

# Save figure without white borders
if save_figs:
    # Build filename with all scenario keys and subkeys
    scenario_str = "_".join(scenario_keys)
    subkeys_str = "_".join([str(subkey) for subkey in settings[scenario_keys[0]].keys()])
    filename = f"{output_dir}{overall_name}_full-load-hours_pv_per-bus_{scenario_str}_{subkeys_str}.pdf"
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    filename_png = filename.replace('.pdf', '.png')
    plt.savefig(filename_png, bbox_inches='tight', dpi=300)

plt.show()

#### Calculate and plot theoretical load hours per bus

In [ ]:
# Calculate theoretical full load hours for solar generators
theoretical_solar_flh = (theoretical_max_per_gen * 1e6).divide(solar_gens_max)  # in hours
# Add theoretical FLH to solar_gens_geo by matching index of theoretical_solar_flh with generator in solar_gens_geo
# Ensure fallback if FLH exists already
if "theoretical_FLH" in solar_gens_geo.columns:
    solar_gens_geo = solar_gens_geo.drop(columns=["theoretical_FLH"])
solar_gens_geo = solar_gens_geo.set_index("generator").join(theoretical_solar_flh.rename("theoretical_FLH")).reset_index()
# solar_gens_geo.head()

In [ ]:
# Plot solar capacity p_nom_opt on map
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"projection": ccrs.PlateCarree()})
# ax.set_title("Solar Generator Capacity (p_nom_opt)", fontsize=20)
# ax.add_feature(cfeature.NaturalEarthFeature('cultural', 'admin_0_countries', '50m'),
#                facecolor='whitesmoke', edgecolor='black', zorder=0)
# # Highlight Algeria
# for country in reader.records():
#     if country.attributes['SOVEREIGNT'] == 'Algeria':
#         ax.add_geometries([country.geometry], crs=ccrs.PlateCarree(),
#                           facecolor='lightgrey', edgecolor='black', linewidth=1, zorder=0)
# ax.set_extent([-11, 15, 18, 38], crs=ccrs.PlateCarree())
# Aggregate p_nom_opt per bus region
region_theoretical_FLH = solar_gens_geo.groupby("bus")["theoretical_FLH"].sum()  # in hours

# Merge with gadm_shapes using GADM ID (partial match)
# Assume gadm_shapes has a column 'GADM_ID' and regions_p_nom_max index contains GADM IDs (possibly with extra info)

def match_gadm_id(row, region_index):
    # Find first region index that contains the GADM ID as substring
    matches = [idx for idx in region_index if row['GADM_ID'] in idx]
    if matches:
        return region_theoretical_FLH[matches[0]]
    else:
        return 0

gadm_regions_selected = gadm_regions['withExp']['400']
gadm_regions_selected["Theoretical FLH"] = gadm_regions_selected.apply(lambda row: match_gadm_id(row, region_theoretical_FLH.index), axis=1)
bus_regions = gadm_regions_selected.copy()

# Plot bus regions colored by FLH
mappable = bus_regions.plot(
    ax=ax,
    column="Theoretical FLH",
    cmap="YlOrRd",
    legend=False,
    edgecolor="black",
    linewidth=0.5,
    zorder=2
)

# Add colorbar (fix: use the first PolyCollection from mappable.collections)
# sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(vmin=bus_regions["FLH"].min(), vmax=bus_regions["FLH"].max()))
sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(vmin=400, vmax=bus_regions["Theoretical FLH"].max()))
sm._A = []  # Dummy array for ScalarMappable
cbar = fig.colorbar(sm, ax=ax, orientation='vertical', fraction=0.036, pad=0.04)
cbar.set_label('Full load hours (FLH)', fontsize=18)
cbar.ax.yaxis.set_major_formatter('{:,.0f}'.format)

plt.tight_layout()

# Ensure the directory exists
output_dir = f"results/graphics_general/sec-NoH2Store/electricity/"
os.makedirs(output_dir, exist_ok=True)

# Save figure without white borders
if save_figs:
    # Build filename with all scenario keys and subkeys
    scenario_str = "_".join(scenario_keys)
    subkeys_str = "_".join([str(subkey) for subkey in settings[scenario_keys[0]].keys()])
    filename = f"{output_dir}{overall_name}_theoretical-full-load-hours_pv_per-bus_{scenario_str}_{subkeys_str}.pdf"
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    filename_png = filename.replace('.pdf', '.png')
    plt.savefig(filename_png, bbox_inches='tight', dpi=300)

# plt.show()

In [ ]:
# Calculate p_nom_opt / p_nom_max ratio
capacity_ratio = solar_gens_opt / solar_gens_max
# Plot capacity ratio of each solar generator
capacity_ratio.sort_values(ascending=False).plot(kind="bar", figsize=(14, 6))
plt.ylabel("Capacity ratio (p_nom_opt / p_nom_max)")
plt.title("Optimized vs Maximum Capacity Ratio per Solar Generator")
plt.ylim(0, 1.1)
plt.tight_layout()
plt.show()

In [ ]:
# Example: pick one generator
gen = "DZ.41_1_AC solar"

# Extract series from all DataFrames in n.generators_t
data = {}
for key, df in n.generators_t.items():
    if gen in df.columns:                # make sure generator exists in that DF
        data[key] = df[gen]

# Combine into a single DataFrame
result = pd.DataFrame(data)

# Add snapshots as first column (index of PyPSA DFs)
result = result.reset_index().rename(columns={"index": "snapshot"})

result